# ABR dataset overview

A quick tour of the four input modalities the resistance models can use, built through the single dataloader in `datasets/`:

| modality | what it is | shape per block |
|---|---|---|
| **dna** | one-hot nucleotides, gene loci concatenated | `(N, 5, L)` — A,C,T,G,gap |
| **protein** | one-hot amino acids, one block per gene | `(N, 20, K)` |
| **biophysical** | MW / pI / hydrophobicity per residue, per gene | `(N, 3, K)` |
| **regulatory** | one-hot promoter/intergenic regions (e.g. *fabG1*) | `(N, 5, L)` |

Runs on small **synthetic** fixtures by default so it works anywhere; flip `USE_REAL = True` on the cluster to point at the real BIG-TB data.

In [1]:
%matplotlib inline
import tempfile, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from datasets import load_dataset, MODALITIES, DRUG_TO_LOCI, DRUG_TO_REGULATORY
from bigtb_ref import REAL_GENOTYPE_DIR, REAL_PHENOTYPE_CSV

USE_REAL = False          # <-- flip to True on Unity (pi_annagreen access)
ALL_MODALITIES = list(MODALITIES)
DRUGS = ['ISONIAZID', 'RIFAMPICIN', 'PYRAZINAMIDE']
print('modalities:', ALL_MODALITIES)

I0000 00:00:1784556155.843701 2851448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784556155.893995 2851448 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784556162.911207 2851448 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


modalities: ['dna', 'protein', 'biophysical', 'regulatory']


In [ ]:
# resolve the data source (real paths, or a synthetic fixture set built once)
if USE_REAL:
    GENO, PHENO, REG = REAL_GENOTYPE_DIR, REAL_PHENOTYPE_CSV, REAL_GENOTYPE_DIR
else:
    from fixtures import build_fixture_dataset
    genes = sorted({g for d in DRUGS for g in DRUG_TO_LOCI[d]})
    regions = sorted({r for d in DRUGS for r in DRUG_TO_REGULATORY.get(d, [])})
    GENO, PHENO = build_fixture_dataset(tempfile.mkdtemp(), genes=genes, drugs=DRUGS,
                                        n_isolates=200, n_codons=60, seed=0,
                                        regulatory_regions=regions)
    REG = GENO
print('source:', 'REAL' if USE_REAL else 'synthetic', '| drugs:', DRUGS)

def blocks_of(data, modality):
    return [b for b in data.blocks if b.modality == modality]

## 1. One drug, every modality

What each modality produces for **isoniazid** (genes *inhA* + *katG*). The regulatory regions are WHO-catalogue-derived per drug; on the real data INH loads *fabG1* (the fabG1–inhA operon promoter), while the synthetic fixtures below generate the full WHO candidate set. One row per feature block = one branch of the model.

In [ ]:
focus = 'ISONIAZID'
data = load_dataset(focus, ALL_MODALITIES, GENO, PHENO, regulatory_dir=REG)
print(f'{focus}: {data.n} isolates | class counts {data.class_counts()}')
print(f'modalities used: {data.modalities}   dropped: {data.dropped}')
pd.DataFrame([{'block': b.name, 'modality': b.modality, 'channels': b.channels,
               'length': b.length, 'note': b.note} for b in data.blocks])

## 2. Characteristics across drugs

Isolate counts, sequence length, and class balance for each drug (DNA modality). `R_frac` is the resistant fraction among labelled isolates — the class-imbalance the loss weighting has to handle.

In [ ]:
recs = []
for d in DRUGS:
    dd = load_dataset(d, ['dna'], GENO, PHENO, regulatory_dir=REG, verbose=False)
    c = dd.class_counts()
    lab = max(c['R'] + c['S'], 1)
    recs.append({'drug': d, 'genes': '+'.join(dd.gene_order), 'n_isolates': dd.n,
                 'dna_len': dd.blocks[0].length, 'R': c['R'], 'S': c['S'],
                 'missing': c['missing'], 'R_frac': round(c['R'] / lab, 3)})
overview = pd.DataFrame(recs)
overview

## 3. Class balance

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(overview))
ax.bar(x, overview['R'], label='resistant', color='#c0504d')
ax.bar(x, overview['S'], bottom=overview['R'], label='susceptible', color='#4f81bd')
ax.bar(x, overview['missing'], bottom=overview['R'] + overview['S'], label='missing', color='#bfbfbf')
ax.set_xticks(x); ax.set_xticklabels(overview['drug'], rotation=20, ha='right')
ax.set_ylabel('isolates'); ax.set_title('Phenotype label balance per drug')
ax.legend(); plt.tight_layout(); plt.show()

## 4. DNA one-hot — what the CNN actually sees

The first stretch of one isolate's one-hot matrix. Each column is a position; the lit row is the base (or the gap channel for an alignment gap).

In [ ]:
dna = blocks_of(data, 'dna')[0].array   # (N, 5, L)
channels = blocks_of(data, 'dna')[0].channel_names
window = dna[0, :, :80]
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.imshow(window, aspect='auto', cmap='Greys', interpolation='nearest')
ax.set_yticks(range(len(channels))); ax.set_yticklabels(channels)
ax.set_xlabel('alignment position'); ax.set_title(f'{focus} DNA one-hot (isolate 0, first 80 bp)')
plt.tight_layout(); plt.show()

## 5. Sequence length & gap content

Per-isolate gap fraction (share of alignment columns that are gaps) — a proxy for indels / partial coverage across the population.

In [ ]:
occupied = dna.sum(axis=1) > 0            # (N, L) columns with any base or gap
gap_frac = dna[:, channels.index('-'), :].sum(axis=1) / np.clip(occupied.sum(axis=1), 1, None)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(gap_frac, bins=30, color='#4f81bd', edgecolor='white')
ax.set_xlabel('gap fraction'); ax.set_ylabel('isolates')
ax.set_title(f'{focus} — per-isolate gap content ({data.n} isolates)')
plt.tight_layout(); plt.show()

## 6. Biophysical traits

Distribution of the three z-scored per-residue properties (over all residues in all isolates) for the first gene's biophysical block.

In [ ]:
bio_block = blocks_of(data, 'biophysical')[0]
bio = bio_block.array                      # (N, 3, K)
resid = (bio != 0).any(axis=1)             # (N, K) real residues (not padding)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for p, (name, ax) in enumerate(zip(bio_block.channel_names, axes)):
    vals = bio[:, p, :][resid]
    ax.hist(vals, bins=25, color='#9bbb59', edgecolor='white')
    ax.set_title(name); ax.set_xlabel('z-score')
axes[0].set_ylabel('residues')
fig.suptitle(f'{focus} — {bio_block.name} biophysical properties')
plt.tight_layout(); plt.show()

## 7. Protein lengths

Translated protein length per isolate for each gene (translation stops at the first stop codon, so nonsense truncations show up as short proteins).

In [ ]:
prot_blocks = blocks_of(data, 'protein')
fig, ax = plt.subplots(figsize=(7, 3.2))
for b in prot_blocks:
    lengths = ((b.array != 0).any(axis=1)).sum(axis=1)   # residues per isolate
    ax.hist(lengths, bins=20, alpha=0.6, label=b.name.split(':')[1], edgecolor='white')
ax.set_xlabel('protein length (residues)'); ax.set_ylabel('isolates')
ax.set_title(f'{focus} — translated protein lengths'); ax.legend()
plt.tight_layout(); plt.show()

---
### Switching to real data

Set `USE_REAL = True` in the config cell and re-run. That points the loader at the Unity `pi_annagreen` paths (`bigtb_ref.REAL_GENOTYPE_DIR` / `REAL_PHENOTYPE_CSV`, ~17.9k isolates). Everything else is identical — the same `load_dataset(drug, modalities, ...)` call.

**Notes for the meeting**
- Regulatory regions are defined per-drug from the **WHO 2023 catalogue** (`datasets/who_catalogue.py`, Tables 21 & 22): the default set is a drug's WHO candidate genes minus its coding loci. On the real data the available ones load — *fabG1* (fabG1–inhA promoter) for isoniazid/ethionamide, *eis* (eis promoter) for kanamycin — the rest are skipped until their FASTAs are curated. Table-22 upstream coordinates + TSS ride along as block metadata.
- Any modality's loci can be chosen explicitly: `load_dataset(..., loci=[...], regulatory_loci=[...])`.
- Biophysical property values are standard published tables (z-scored), a stand-in until we confirm Kulkarni et al.'s exact table (see also `biophysical_properties_rdkit.ipynb` for RDKit-derived values).
- Synthetic numbers here are meaningless by construction; this notebook is about *shapes and coverage*, not performance.